# Phase 2.2 — Bounded Hyperparameter Tuning

Bounded, one-shot tuning pass over the Phase 2.1 production models
(Isolation Forest + Autoencoder ensemble). Runs the specified grids
only, reports all results, and adopts the best config **only if it
beats the Phase 2.1 baseline ensemble AUC-ROC of 0.5912**. No
additional configs are added beyond what's listed in each cell.

In [1]:
import sys
from pathlib import Path

import joblib
import numpy as np
import torch

BACKEND_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(BACKEND_DIR) not in sys.path:
    sys.path.insert(0, str(BACKEND_DIR))

from core.model import Autoencoder, FEATURE_ORDER, get_cleaned_train_test_data

# Reuse the cleanlab-cleaned legit-only training split and the
# untouched test split from Phase 2.1 -- never recomputed here.
X_train_cleaned, y_train_cleaned, X_test, y_test = get_cleaned_train_test_data()
X_train_legit = X_train_cleaned[y_train_cleaned == 0]

scaler = joblib.load(BACKEND_DIR / "trained_models" / "scaler.pkl")

# Scaler-transformed versions for the autoencoder (IF stays unscaled --
# tree-based, scale-invariant).
X_train_legit_scaled = scaler.transform(X_train_legit)
X_test_scaled = scaler.transform(X_test)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"backend dir on path: {BACKEND_DIR}")
print(f"X_train_legit shape (unscaled, IF input): {X_train_legit.shape}")
print(f"X_train_legit_scaled shape (AE input):     {X_train_legit_scaled.shape}")
print(f"X_test shape: {X_test.shape}, X_test_scaled shape: {X_test_scaled.shape}")
print(f"test cheater rate: {y_test.mean():.4f}")
print(f"Using device: {device}")

backend dir on path: D:\ARGUS\backend
X_train_legit shape (unscaled, IF input): (7610, 11)
X_train_legit_scaled shape (AE input):     (7610, 11)
X_test shape: (2400, 11), X_test_scaled shape: (2400, 11)
test cheater rate: 0.1667
Using device: cuda


## Isolation Forest grid (3x3 = 9 configs)

In [2]:
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score

if_grid_results = []
for n_estimators in [100, 200, 300]:
    for max_features in [0.5, 0.7, 1.0]:
        model = IsolationForest(
            n_estimators=n_estimators,
            max_features=max_features,
            contamination="auto",
            random_state=42,
        )
        model.fit(X_train_legit)

        anomaly_scores = -model.decision_function(X_test)
        auc = roc_auc_score(y_test, anomaly_scores)

        if_grid_results.append({
            "n_estimators": n_estimators,
            "max_features": max_features,
            "auc": auc,
            "model": model,
        })

if_grid_sorted = sorted(if_grid_results, key=lambda r: r["auc"], reverse=True)

print(f"{'n_estimators':>12} {'max_features':>12} {'AUC-ROC':>10}")
print("-" * 36)
for r in if_grid_sorted:
    print(f"{r['n_estimators']:>12} {r['max_features']:>12} {r['auc']:>10.4f}")

best_if = if_grid_sorted[0]
print()
print(f"Best IF config: n_estimators={best_if['n_estimators']}, "
      f"max_features={best_if['max_features']}, AUC-ROC={best_if['auc']:.4f}")

n_estimators max_features    AUC-ROC
------------------------------------
         300          0.5     0.5941
         300          1.0     0.5929
         100          1.0     0.5911
         100          0.5     0.5910
         200          1.0     0.5904
         200          0.5     0.5899
         300          0.7     0.5887
         200          0.7     0.5856
         100          0.7     0.5852

Best IF config: n_estimators=300, max_features=0.5, AUC-ROC=0.5941


## Autoencoder grid (3x2 = 6 configs)\n\nBounded to exactly 6 configs (`bottleneck_dim` x `learning_rate`). `torch.manual_seed(42)` reset before each config for comparability.

In [3]:
import torch.nn as nn
from sklearn.model_selection import train_test_split as _tts
from torch.utils.data import DataLoader, TensorDataset

MAX_EPOCHS = 200
PATIENCE = 15

ae_grid_configs = [
    (bottleneck_dim, lr)
    for bottleneck_dim in [4, 5, 6]
    for lr in [1e-3, 5e-4]
]

ae_grid_results = []

for bottleneck_dim, lr in ae_grid_configs:
    torch.manual_seed(42)

    X_ae_train, X_ae_val = _tts(X_train_legit_scaled, test_size=0.10, random_state=42)
    train_tensor = torch.tensor(X_ae_train, dtype=torch.float32)
    val_tensor = torch.tensor(X_ae_val, dtype=torch.float32).to(device)
    train_loader = DataLoader(TensorDataset(train_tensor), batch_size=128, shuffle=True)

    model = Autoencoder(input_dim=11, hidden_dim=8, bottleneck_dim=bottleneck_dim).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    best_val_loss = float("inf")
    best_state = None
    epochs_without_improvement = 0
    epochs_run = MAX_EPOCHS

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        for (batch,) in train_loader:
            batch = batch.to(device)
            optimizer.zero_grad()
            reconstruction = model(batch)
            loss = criterion(reconstruction, batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_reconstruction = model(val_tensor)
            epoch_val_loss = criterion(val_reconstruction, val_tensor).item()

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= PATIENCE:
            epochs_run = epoch
            break

    model.load_state_dict(best_state)

    model.eval()
    test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
    with torch.no_grad():
        test_reconstruction = model(test_tensor)
        per_sample_mse = torch.mean((test_reconstruction - test_tensor) ** 2, dim=1).cpu().numpy()

    score_min, score_max = per_sample_mse.min(), per_sample_mse.max()
    ae_scores_normalized = (per_sample_mse - score_min) / (score_max - score_min)
    auc = roc_auc_score(y_test, ae_scores_normalized)

    ae_grid_results.append({
        "bottleneck_dim": bottleneck_dim,
        "learning_rate": lr,
        "epochs_run": epochs_run,
        "final_val_loss": best_val_loss,
        "auc": auc,
        "state_dict": best_state,
    })

    print(f"bottleneck_dim={bottleneck_dim} lr={lr}: epochs_run={epochs_run} "
          f"final_val_loss={best_val_loss:.6f} AUC-ROC={auc:.4f}")

ae_grid_sorted = sorted(ae_grid_results, key=lambda r: r["auc"], reverse=True)

print()
print(f"{'bottleneck_dim':>15} {'learning_rate':>15} {'epochs_run':>11} "
      f"{'final_val_loss':>15} {'AUC-ROC':>10}")
print("-" * 72)
for r in ae_grid_sorted:
    print(f"{r['bottleneck_dim']:>15} {r['learning_rate']:>15} {r['epochs_run']:>11} "
          f"{r['final_val_loss']:>15.6f} {r['auc']:>10.4f}")

best_ae = ae_grid_sorted[0]
print()
print(f"Best AE config: bottleneck_dim={best_ae['bottleneck_dim']}, "
      f"learning_rate={best_ae['learning_rate']}, AUC-ROC={best_ae['auc']:.4f}")

bottleneck_dim=4 lr=0.001: epochs_run=200 final_val_loss=0.276788 AUC-ROC=0.5892


bottleneck_dim=4 lr=0.0005: epochs_run=200 final_val_loss=0.209106 AUC-ROC=0.5869


bottleneck_dim=5 lr=0.001: epochs_run=200 final_val_loss=0.293522 AUC-ROC=0.6003


bottleneck_dim=5 lr=0.0005: epochs_run=200 final_val_loss=0.311849 AUC-ROC=0.5779


bottleneck_dim=6 lr=0.001: epochs_run=109 final_val_loss=0.132998 AUC-ROC=0.5355


bottleneck_dim=6 lr=0.0005: epochs_run=112 final_val_loss=0.133223 AUC-ROC=0.5396

 bottleneck_dim   learning_rate  epochs_run  final_val_loss    AUC-ROC
------------------------------------------------------------------------
              5           0.001         200        0.293522     0.6003
              4           0.001         200        0.276788     0.5892
              4          0.0005         200        0.209106     0.5869
              5          0.0005         200        0.311849     0.5779
              6          0.0005         112        0.133223     0.5396
              6           0.001         109        0.132998     0.5355

Best AE config: bottleneck_dim=5, learning_rate=0.001, AUC-ROC=0.6003


## Ensemble weight tuning\n\nUsing the best IF config (Cell 2) and best AE config (Cell 3), score X_test with both and sweep the ensemble weight.

In [4]:
best_if_model = best_if["model"]
iso_raw_scores = -best_if_model.decision_function(X_test)
iso_min, iso_max = iso_raw_scores.min(), iso_raw_scores.max()
if_scores_normalized = (iso_raw_scores - iso_min) / (iso_max - iso_min)

best_ae_model = Autoencoder(
    input_dim=11, hidden_dim=8, bottleneck_dim=best_ae["bottleneck_dim"]
).to(device)
best_ae_model.load_state_dict(best_ae["state_dict"])
best_ae_model.eval()

test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32).to(device)
with torch.no_grad():
    test_reconstruction = best_ae_model(test_tensor)
    per_sample_mse = torch.mean((test_reconstruction - test_tensor) ** 2, dim=1).cpu().numpy()

score_min, score_max = per_sample_mse.min(), per_sample_mse.max()
ae_scores_normalized = (per_sample_mse - score_min) / (score_max - score_min)

weight_results = []
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    ensemble_score = w * if_scores_normalized + (1 - w) * ae_scores_normalized
    auc = roc_auc_score(y_test, ensemble_score)
    weight_results.append({"weight": w, "auc": auc})

weight_sorted = sorted(weight_results, key=lambda r: r["auc"], reverse=True)

print(f"{'weight (IF)':>12} {'AUC-ROC':>10}")
print("-" * 24)
for r in weight_sorted:
    print(f"{r['weight']:>12} {r['auc']:>10.4f}")

best_weight = weight_sorted[0]
print()
print(f"Best ensemble weight: w={best_weight['weight']} (IF weight), "
      f"AUC-ROC={best_weight['auc']:.4f}")

 weight (IF)    AUC-ROC


------------------------
         0.3     0.5987
         0.4     0.5973
         0.5     0.5963
         0.6     0.5956
         0.7     0.5951

Best ensemble weight: w=0.3 (IF weight), AUC-ROC=0.5987


## Final comparison: Phase 2.1 baseline vs Phase 2.2 tuned

In [5]:
BASELINE_IF = 0.5904
BASELINE_AE = 0.5643
BASELINE_ENSEMBLE = 0.5912

if_config_str = f"n_estimators={best_if['n_estimators']}, max_features={best_if['max_features']}"
ae_config_str = f"bottleneck_dim={best_ae['bottleneck_dim']}, lr={best_ae['learning_rate']}"
weight_config_str = f"w={best_weight['weight']} (IF weight)"

print("=" * 78)
print("PHASE 2.1 BASELINE vs PHASE 2.2 TUNED")
print("=" * 78)
print(f"{'Model':<28} {'Config':<32} {'AUC-ROC':>10}")
print("-" * 78)
print(f"{'Phase 2.1 IF baseline':<28} {'n_estimators=200 (default)':<32} {BASELINE_IF:>10.4f}")
print(f"{'Phase 2.1 AE baseline':<28} {'bottleneck_dim=5, lr=1e-3':<32} {BASELINE_AE:>10.4f}")
print(f"{'Phase 2.1 ensemble baseline':<28} {'50/50':<32} {BASELINE_ENSEMBLE:>10.4f}")
print("-" * 78)
print(f"{'Phase 2.2 best IF':<28} {if_config_str:<32} {best_if['auc']:>10.4f}")
print(f"{'Phase 2.2 best AE':<28} {ae_config_str:<32} {best_ae['auc']:>10.4f}")
print(f"{'Phase 2.2 best ensemble':<28} {weight_config_str:<32} {best_weight['auc']:>10.4f}")
print("=" * 78)

delta = best_weight["auc"] - BASELINE_ENSEMBLE
print()
print(f"Delta (Phase 2.2 best ensemble - Phase 2.1 baseline ensemble): {delta:+.4f}")
if delta > 0:
    print(f"Tuning IMPROVED the ensemble by {delta:.4f} AUC-ROC over the Phase 2.1 baseline.")
else:
    print(f"Tuning did NOT improve the ensemble (delta {delta:+.4f}). "
          f"Phase 2.1 config remains production.")

PHASE 2.1 BASELINE vs PHASE 2.2 TUNED
Model                        Config                              AUC-ROC
------------------------------------------------------------------------------
Phase 2.1 IF baseline        n_estimators=200 (default)           0.5904
Phase 2.1 AE baseline        bottleneck_dim=5, lr=1e-3            0.5643
Phase 2.1 ensemble baseline  50/50                                0.5912
------------------------------------------------------------------------------
Phase 2.2 best IF            n_estimators=300, max_features=0.5     0.5941
Phase 2.2 best AE            bottleneck_dim=5, lr=0.001           0.6003
Phase 2.2 best ensemble      w=0.3 (IF weight)                    0.5987

Delta (Phase 2.2 best ensemble - Phase 2.1 baseline ensemble): +0.0075
Tuning IMPROVED the ensemble by 0.0075 AUC-ROC over the Phase 2.1 baseline.


## Save best configuration (only if it beats the 0.5912 baseline)

In [6]:
import json

MODELS_DIR = BACKEND_DIR / "trained_models"

if best_weight["auc"] > BASELINE_ENSEMBLE:
    joblib.dump(best_if_model, MODELS_DIR / "isolation_forest.pkl")
    torch.save(best_ae["state_dict"], MODELS_DIR / "autoencoder.pt")

    config = {
        "input_dim": 11,
        "hidden_dim": 8,
        "bottleneck_dim": best_ae["bottleneck_dim"],
        "feature_order": FEATURE_ORDER,
    }
    with open(MODELS_DIR / "autoencoder_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    print(f"Phase 2.2 tuned ensemble ({best_weight['auc']:.4f}) beat the Phase 2.1 "
          f"baseline ({BASELINE_ENSEMBLE:.4f}) -- ADOPTED as production.")
    print(f"Overwrote isolation_forest.pkl: n_estimators={best_if['n_estimators']}, "
          f"max_features={best_if['max_features']}")
    print(f"Overwrote autoencoder.pt: bottleneck_dim={best_ae['bottleneck_dim']}, "
          f"lr={best_ae['learning_rate']}")
    print(f"Updated autoencoder_config.json: {config}")
    print(f"Production ensemble weight: w={best_weight['weight']} (IF weight)")
else:
    print(f"Phase 2.2 tuned ensemble ({best_weight['auc']:.4f}) did NOT beat the "
          f"Phase 2.1 baseline ({BASELINE_ENSEMBLE:.4f}).")
    print("Phase 2.1 files (isolation_forest.pkl, autoencoder.pt, "
          "autoencoder_config.json) left UNTOUCHED.")
    print("Phase 2.1 configuration (IF n_estimators=200 default, "
          "AE bottleneck_dim=5 lr=1e-3, 50/50 ensemble) REMAINS PRODUCTION.")

Phase 2.2 tuned ensemble (0.5987) beat the Phase 2.1 baseline (0.5912) -- ADOPTED as production.
Overwrote isolation_forest.pkl: n_estimators=300, max_features=0.5
Overwrote autoencoder.pt: bottleneck_dim=5, lr=0.001
Updated autoencoder_config.json: {'input_dim': 11, 'hidden_dim': 8, 'bottleneck_dim': 5, 'feature_order': ['peak_yaw_delta', 'peak_pitch_delta', 'mean_yaw_delta', 'snap_count', 'min_cv_yaw', 'min_cv_pitch', 'cv_yaw_std', 'cv_pitch_std', 'fire_on_target_rate', 'yaw_jerk', 'engagement_firing_rate']}
Production ensemble weight: w=0.3 (IF weight)
